In [102]:
from pathlib import Path

import pandas as pd
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from catboost import CatBoostClassifier
import numpy as np

from thresholding import ThresholdedClassifier, choose_best_threshold_for_recall


# Load data
X_train = pd.read_csv(Path(r"data/X_train_selected.csv"))
X_test = pd.read_csv(Path(r"data/X_test_selected.csv"))
y_train = pd.read_csv(Path(r"data/y_train.csv")).squeeze("columns")
y_test = pd.read_csv(Path(r"data/y_test.csv")).squeeze("columns")


In [104]:
# Model
logreg_model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight="balanced"
)

In [105]:
# Model
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

In [106]:
# Model
xgb_model = XGBClassifier(
    n_estimators=450,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

In [107]:
#Model 
cat_boost = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    eval_metric="Logloss",
    random_seed=42,
    verbose=False
)

In [108]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}


In [109]:
logreg_cv = cross_validate(
    logreg_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Aggregate results
logreg_results = pd.DataFrame({
    "metric": list(scoring.keys()),
    "mean": [np.mean(logreg_cv[f"test_{m}"]) for m in scoring.keys()],
    "std": [np.std(logreg_cv[f"test_{m}"]) for m in scoring.keys()]
})

print("Logistic Regression - Cross-Validation Results")
display(logreg_results)

Logistic Regression - Cross-Validation Results


,metric,mean,std
0,accuracy,0.792073,0.005176
1,precision,0.659376,0.007580
2,recall,0.704085,0.013803
3,f1,0.680946,0.009110


In [110]:
# CV
rf_cv = cross_validate(
    rf_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Aggregate results
rf_results = pd.DataFrame({
    "metric": list(scoring.keys()),
    "mean": [np.mean(rf_cv[f"test_{m}"]) for m in scoring.keys()],
    "std": [np.std(rf_cv[f"test_{m}"]) for m in scoring.keys()]
})

print("Random Forest - Cross-Validation Results")
display(rf_results)

Random Forest - Cross-Validation Results


,metric,mean,std
0,accuracy,0.882813,0.004108
1,precision,0.936582,0.004295
2,recall,0.673834,0.011984
3,f1,0.783723,0.008980


In [111]:
# CV
cat_boost_cv = cross_validate(
    cat_boost,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Aggregate results
cat_boost_results = pd.DataFrame({
    "metric": list(scoring.keys()),
    "mean": [np.mean(cat_boost_cv[f"test_{m}"]) for m in scoring.keys()],
    "std": [np.std(cat_boost_cv[f"test_{m}"]) for m in scoring.keys()]
})

print("CatBoost - Cross-Validation Results")
display(cat_boost_results)

CatBoost - Cross-Validation Results


,metric,mean,std
0,accuracy,0.883930,0.002785
1,precision,0.929625,0.004262
2,recall,0.683524,0.009046
3,f1,0.787763,0.006157


In [112]:
# CV
xgb_cv = cross_validate(
    xgb_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Aggregate results
xgb_results = pd.DataFrame({
    "metric": list(scoring.keys()),
    "mean": [np.mean(xgb_cv[f"test_{m}"]) for m in scoring.keys()],
    "std": [np.std(xgb_cv[f"test_{m}"]) for m in scoring.keys()]
})

print("XGBoost - Cross-Validation Results")
display(xgb_results)

XGBoost - Cross-Validation Results


,metric,mean,std
0,accuracy,0.883483,0.003637
1,precision,0.916850,0.005592
2,recall,0.693214,0.009854
3,f1,0.789468,0.007474


## Vers?o Final fitted no total do train dataset


In [113]:
threshold_grid = np.arange(0.30, 0.91, 0.05)

base_models = {
    "Logistic Regression": logreg_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
    "CatBoost": cat_boost,
}

best_thresholds = {}
threshold_rows = []
wrapped_models = {}

for model_name, model in base_models.items():
    # Out-of-fold probabilities avoid tuning thresholds on the held-out test set.
    y_prob_oof = cross_val_predict(
        clone(model),
        X_train,
        y_train,
        cv=cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    best_threshold, best_recall, best_f1 = choose_best_threshold_for_recall(
        y_train,
        y_prob_oof,
        threshold_grid,
    )

    best_thresholds[model_name] = best_threshold
    threshold_rows.append({
        "model": model_name,
        "best_threshold": best_threshold,
        "oof_recall": best_recall,
        "oof_f1": best_f1,
    })

    fitted_model = clone(model).fit(X_train, y_train)
    wrapped_models[model_name] = ThresholdedClassifier(
        base_model=fitted_model,
        threshold=best_threshold,
    )

thresholds_df = pd.DataFrame(threshold_rows).sort_values("oof_recall", ascending=False)
display(thresholds_df)

logreg_model = wrapped_models["Logistic Regression"]
rf_model = wrapped_models["Random Forest"]
xgb_model = wrapped_models["XGBoost"]
cat_boost = wrapped_models["CatBoost"]


,model,best_threshold,oof_recall,oof_f1
0,Logistic Regression,0.3,0.862917,0.595790
1,Random Forest,0.3,0.776885,0.778264
3,CatBoost,0.3,0.767667,0.785205
2,XGBoost,0.3,0.766013,0.782472


In [114]:
from pathlib import Path

import pandas as pd
import joblib


# Save fitted threshold-aware models
models_dir = Path("models")
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(logreg_model, models_dir / "logreg_model.joblib")
joblib.dump(rf_model, models_dir / "rf_model.joblib")
joblib.dump(xgb_model, models_dir / "xgb_model.joblib")
joblib.dump(cat_boost, models_dir / "cat_boost.joblib")

thresholds_path = models_dir / "model_thresholds.csv"
thresholds_df.to_csv(thresholds_path, index=False)

print("Threshold-aware models saved successfully.")
print(f"Saved thresholds to {thresholds_path}")


Threshold-aware models saved successfully.
Saved thresholds to models\model_thresholds.csv
